In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.warn("Ignore")
import geopandas as gpd
from datetime import datetime, timedelta
%matplotlib inline
from matplotlib import colors as c
from mpl_toolkits.basemap import Basemap, shiftgrid
import os
os.environ['USE_PYGEOS'] = '0'
os.getcwd()
os.chdir("C:\\Users")
data = pd.read_csv("C:\\Users\\Consolidated.csv")
data.head()
data.columns
data["st_tm"] = data["Start_Time"]
i = 0
for i in range(len(data)):
    if pd.isnull(data["st_tm"][i]):
        data["st_tm"][i] = data["st_tm"][i-1] + \
        (data["Time (s)"][i] - data["Time (s)"][i-1])
    else:
        pass
data["Start_Time"] = data.apply(lambda x : convert_time(x['st_tm']), axis=1)
data[(data["TripNum"]==1) & (data["User"]==1)].tail()
data = data.replace(np.nan, 0)
data = data.replace("#NUM!", 0)
data["Speed (m/s)"] = pd.to_numeric(data["Speed (m/s)"])
data.dtypes
data["Speed_kph"] = ''
def speed_conversion(speed_ms):
    kph_speed = (speed_ms *3600)/1000
    return kph_speed
data["Speed_kph"] = data.apply(lambda x : speed_conversion(x['Speed (m/s)']), axis=1)
data.head(5)
def convert_time(timestamp):
    dt_obj = datetime.fromtimestamp(timestamp)
    return dt_obj


    
data["Start_Time"] = data.apply(lambda x : convert_time(x['st_tm']), axis=1)
data['trip_id'] = 'User ' + data['User'].astype(str) + '- Trip ' + \
data['TripNum'].astype(str)
data['trip_id'].nunique()
data["harsh_dec"], data["harsh_acc"] = '', ''
data['speed_diff'] = data['Speed_kph'].diff()
data1 = data.groupby(["trip_id"]).max("Distance (km)")
data1["Distance (km)"].sum()
for i in range(len(data)):
    if data["speed_diff"].iloc[i] > 10:
        data["harsh_acc"].iloc[i] = 1
    elif data["speed_diff"].iloc[i] < -10:
        data["harsh_dec"].iloc[i] = 1
data["harsh_acc"] = data["harsh_acc"].replace('', 0)
data["harsh_dec"] = data["harsh_dec"].replace('', 0)
data["dow"] = ''
for i in range(len(data)):
    x = data["Date"].iloc[i].split(", ")
    data["dow"].iloc[i] = x[0]
from datetime import datetime, timedelta
import datetime as dt
data.dtypes
i=0
data["hour"] =''
for i in range(len(data)):
    data["hour"].iloc[i] = data["Start_Time"].iloc[i].hour
data.head()
data2 = data.groupby(["trip_id", "Driver Type"]).\
    agg(trip_duration=('Time (s)',np.max),
    trip_distance=('Distance (km)',np.max),
    max_speed=('Speed_kph',np.max), 
    harsh_acc = ("harsh_acc", np.sum),
    harsh_dec = ("harsh_dec", np.sum))
data2["duration_mins"] = round(data2["trip_duration"]/60,0)
bins=[0,20,60,np.inf]
names=['0-20 Mins','21-60 Mins','>60 Mins']
data2["trip_bucket"]=pd.cut(data2["trip_duration"],bins,labels=names)
data2["hapm"] = data2["harsh_acc"]/data2["duration_mins"]
data2["hdpm"] = data2["harsh_dec"]/data2["duration_mins"]
data2["harsh_events"] = data2["harsh_acc"] + data2["harsh_dec"]
data2["harsh_events_pm"] = data2["hapm"] + data2["hdpm"]
data2["trip_distance"].max()
data2.head(2)





data3 = data2[["harsh_events", "trip_distance"]]
data4 = data3.dropna(inplace=False)
data4.head()
data4.shape
X = data4.iloc[:, [0, 1]].values
!pip show threadpoolctl
import scipy.cluster.hierarchy as sch
dendrogram = sch.dendrogram(sch.linkage(X, method = 'ward'))
plt.title('Dendrogram')
plt.xlabel('Harsh Events')
plt.ylabel('Euclidean distances')
plt.show()
from sklearn.cluster import AgglomerativeClustering
hc = AgglomerativeClustering(n_clusters = 3, metric = 'euclidean', linkage = 'ward')
y_hc = hc.fit_predict(X)
plt.scatter(X[y_hc == 0, 0], X[y_hc == 0, 1], s = 100, c = 'red', label = 'Cluster 1')
plt.scatter(X[y_hc == 1, 0], X[y_hc == 1, 1], s = 100, c = 'blue', label = 'Cluster 2')
plt.scatter(X[y_hc == 2, 0], X[y_hc == 2, 1], s = 100, c = 'green', label = 'Cluster 3')
plt.title('Clusters of drivers')
plt.xlabel('Harsh Events')
plt.ylabel('Trip Distance')
plt.legend()
plt.show()
from sklearn.cluster import AgglomerativeClustering
hc = AgglomerativeClustering(n_clusters = 4, metric = 'euclidean', linkage = 'ward')
y_hc = hc.fit_predict(X)
plt.scatter(X[y_hc == 0, 0], X[y_hc == 0, 1], s = 100, c = 'red', label = 'Cluster 1')
plt.scatter(X[y_hc == 1, 0], X[y_hc == 1, 1], s = 100, c = 'blue', label = 'Cluster 2')
plt.scatter(X[y_hc == 2, 0], X[y_hc == 2, 1], s = 100, c = 'green', label = 'Cluster 3')
plt.scatter(X[y_hc == 3, 0], X[y_hc == 3, 1], s = 100, c = 'cyan', label = 'Cluster 4')
plt.title('Clusters of drivers')
plt.xlabel('Driving time (mins)')
plt.ylabel('Harsh events')
plt.legend()
plt.show()






from sklearn.cluster import AgglomerativeClustering
hc = AgglomerativeClustering(n_clusters = 3, metric = 'euclidean', linkage = 'ward')
y_hc = hc.fit_predict(X)
plt.scatter(X[y_hc == 0, 0], X[y_hc == 0, 1], s = 100, c = 'blue', label = 'Calm')
plt.scatter(X[y_hc == 2, 0], X[y_hc == 2, 1], s = 100, c = 'green', label = 'Rational')
plt.scatter(X[y_hc == 1, 0], X[y_hc == 1, 1], s = 100, c = 'red', label = 'Aggressive')
plt.title('Driving behaviour clusters')
plt.xlabel('Driving time (mins)')
plt.ylabel('Harsh events')
plt.legend()
plt.show()
from sklearn.cluster import AgglomerativeClustering
hc = AgglomerativeClustering(n_clusters = 2, metric = 'euclidean', linkage = 'ward')
y_hc = hc.fit_predict(X)
plt.scatter(X[y_hc == 0, 0], X[y_hc == 0, 1], s = 100, c = 'red', label = 'Aggressive')
plt.scatter(X[y_hc == 1, 0], X[y_hc == 1, 1], s = 100, c = 'blue', label = 'Rational')
plt.title('Driving behaviour clusters')
plt.xlabel('Driving time (mins)')
plt.ylabel('Harsh events')
plt.legend()
plt.show()
df_test = data[(data["User"] == 1) & (data["TripNum"] == 12)]
df_test2 = data[(data["User"] == 3) & (data["TripNum"] == 1)]
df_test.head()
df_test2 = data[data["Driver Type"] == "Driver"]
plt.figure(figsize = (20,5))
sns.lineplot(x = "Time (s)", y = "Speed_kph", data = df_test)
sns.lineplot(x = "Time (s)", y = "Speed_kph", data = df_test2)
df_test = data[(data["User"] == 1) & (data["TripNum"] == 1)]
df_test2 = data[(data["User"] == 3) & (data["TripNum"] == 1)]
plt.figure(figsize = (20,5))
sns.lineplot(x = "Time (s)", y = "Speed_kph", data = df_test)
sns.lineplot(x = "Time (s)", y = "Speed_kph", data = df_test2)
df_test = data[(data["User"] == 1) & (data["TripNum"] == 4)]
plt.figure(figsize = (20,5))
sns.lineplot(x = "Time (s)", y = "Speed_kph", data = df_test)





import shapely.wkb as wkblib
import osmium
class StreetsHandler(osmium.SimpleHandler):
    def __init__(self):
        osmium.SimpleHandler.__init__(self)
        self.num_nodes = 0
        self.num_relations = 0
        self.num_ways = 0
        self.street_relations = []
        self.street_relation_members = []
        self.street_ways = []
        self.wkbfab = osmium.geom.WKBFactory()
    def way(self, w):
        if w.tags.get("highway") is not None :
            try:
                wkb = self.wkbfab.create_linestring(w)
                geo = wkblib.loads(wkb, hex=True)
            except :
                return
            row = {"w_id": w.id, "geo": geo }
            for key, value in w.tags:
                row[key] = value 
            self.street_ways.append(row)
            self.num_ways += 1
    def relation(self, r):
        if r.tags.get("type") == "associatedStreet" and r.tags.get("name") is not None:
            row = { "r_id": r.id }
            for key, value in r.tags: 
                row[key] = value
            self.street_relations.append(row)
            for member in r.members:
                self.street_relation_members.append({
                    "r_id": r.id, "ref": member.ref,
                    "role": member.role, "type": member.type, })
                self.num_relations+= 1
handler = StreetsHandler()
osm_file = "northern-zone-latest.osm.pbf"
handler.apply_file(osm_file, locations = True, idx = 'flex_mem')
%time
print(f"num ways : {handler.num_ways}")






%time
osm_tn = "tennessee-latest.osm.pbf"
handler.apply_file(osm_file, locations = True, idx = 'flex_mem')
print(f"num ways : {handler.num_ways}")
%time
street_ways_tn = pd.DataFrame(handler.street_ways)
%time
street_ways = pd.DataFrame(handler.street_ways)
street_ways.shape
street_ways.columns
street_ways["highway"].unique()
features = ['w_id', 'geo', 'highway', 'name', 'oneway', 'lanes', 'maxspeed', \
            'surface', 'width', 'motorroad', 'junction', 'smoothness', \
            'service', 'maxspeed:hgv', 'trail_visibility', 'living_street']
shapefile = street_ways[features]
shapefile_tn = street_ways_tn[features]
import geopandas as gpd
import pygeos
data.columns
geo_data = data.replace(" (°)", "", inplace=False)
geo_data = data.rename(columns={'Latitude (°)': 'latitude', 'Longitude (°)': 'longitude',
                               'Altitude (m)': 'altitude', 'Altitude WGS84 (m)': 'altitude_wgs84',
                                'Speed (m/s)': 'speed_ms', 'Direction (°)': 'distance', 
                                'Distance (km)': 'distance_km', 'Horizontal Accuracy (m)': 'h_accuracy', 
                                'Vertical Accuracy (m)': 'v_accuracy'})
geo_data.head(2)
gdf = gpd.GeoDataFrame(geo_data, geometry=gpd.points_from_xy\
                       (geo_data.longitude, geo_data.latitude), crs="EPSG:4326")
gdf.crs
shapefile.head(2)
shapefile_gdf = gpd.GeoDataFrame(shapefile, geometry=shapefile.geo, crs="EPSG:4326")
shapefile_tn_gdf = gpd.GeoDataFrame(shapefile_tn, geometry=shapefile_tn.geo, crs="EPSG:4326")
%%time
shapefile_tn_gdf.crs
shapefile_tn_gdf = shapefile_tn_gdf.to_crs("EPSG:4326")
shapefile_gdf = shapefile_gdf.to_crs("EPSG:4326")
shapefile_dc_gdf = shapefile_dc_gdf.to_crs("EPSG:4326")
shapefile_tn_gdf = shapefile_tn_gdf.to_crs("EPSG:4326")
gdf = gdf.to_crs("EPSG:4326")
gdf.head()





%%time
join = gpd.sjoin(gdf, shapefile_gdf, how="inner", predicate="contains")
join_left_df = gdf.sjoin(shapefile_gdf, how="left", predicate = "within")
join_left_df
join_left_df = gdf.sjoin_nearest(shapefile_gdf, how="left", distance_col="Distance")
distance_col="Distances"
join_left_df.head(2)
join_left_df.to_csv("Test_output_0429.csv", index= False)
df_updated = join_left_df.copy()
%%time
df_updated['maxspeed'] = df_updated.apply(lambda row: 30 if (pd.isna(row['maxspeed']) & \
                        (row["highway"] == "residential")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 30 if (pd.isna(row['maxspeed']) & \
                        (row["highway"] == "primary_link")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 50 if (pd.isna(row['maxspeed']) & 
                        \(row["highway"] == "primary")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 80 if (pd.isna(row['maxspeed']) & 
                        \(row["highway"] == "trunk")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 50 if (pd.isna(row['maxspeed']) & 
                        \(row["highway"] == "trunk_link")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 20 if (pd.isna(row['maxspeed']) & \
                        (row["highway"] == "service")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 80 if (pd.isna(row['maxspeed']) & \
                        (row["highway"] == "motorway")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 50 if (pd.isna(row['maxspeed']) & \
                        (row["highway"] == "motorway_link")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 30 if (pd.isna(row['maxspeed']) & \
                        (row["highway"] == "tertiary")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 20 if (pd.isna(row['maxspeed']) & \
                        (row["highway"] == "tertiary_link")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 50 if (pd.isna(row['maxspeed']) & \
                        (row["highway"] == "unclassified")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 50 if (pd.isna(row['maxspeed']) & \
                        (row["highway"] == "secondary")) else row['maxspeed'],axis=1)
df_updated['maxspeed'] = df_updated.apply(lambda row: 40 if (pd.isna(row['maxspeed']) & \
                        (row["highway"] == "secondary_link")) else row['maxspeed'],axis=1)
df_updated["maxspeed"] = df_updated['maxspeed'].fillna(0).astype(int)
df_test = df_updated[(df_updated["User"] == 1) & (df_updated["TripNum"] == 4)]
df_test2 = df_updated[(df_updated["User"] == 1) & (df_updated["TripNum"] == 4)]
plt.figure(figsize = (20,5))
sns.lineplot(x = "Time (s)", y = "Speed_kph", data = df_test, legend="brief", label = "actual_speed")
sns.lineplot(x = "Time (s)", y = "maxspeed", data = df_test2, legend="brief", label = "speed_limit")
df_test = df_updated[(df_updated["User"] == 5) & (df_updated["TripNum"] == 1)]
df_test2 = df_updated[(df_updated["User"] == 5) & (df_updated["TripNum"] == 1)]
plt.figure(figsize = (20,5))
sns.lineplot(x = "Time (s)", y = "Speed_kph", data = df_test, legend="brief", label = "actual_speed")
sns.lineplot(x = "Time (s)", y = "maxspeed", data = df_test2, legend="brief", label = "speed_limit")
df_test = df_updated[(df_updated["User"] == 1) & (df_updated["TripNum"] == 7)]
df_test2 = df_updated[(df_updated["User"] == 1) & (df_updated["TripNum"] == 7)]
plt.figure(figsize = (20,5))
sns.lineplot(x = "Time (s)", y = "Speed_kph", data = df_test, legend="brief", label = "actual_speed")
sns.lineplot(x = "Time (s)", y = "maxspeed", data = df_test2, legend="brief", label = "speed_limit")






df_updated['overspeed'] = df_updated.apply(lambda row: 1 if \
(row['maxspeed']!=0 and (row["Speed_kph"] - row["maxspeed"])>0) else 0, axis=1)
df_updated.columns
data2 = df_updated.groupby(["trip_id", "Driver Type"]).\
    agg(trip_duration=('Time (s)',np.max),
                                                           trip_distance=('distance_km',np.max),
                                                           max_speed=('Speed_kph',np.max), 
                                                           overspeed = ("overspeed", np.sum),
                                                           harsh_acc = ("harsh_acc", np.sum),
                                                           harsh_dec = ("harsh_dec", np.sum))
data2["duration_mins"] = round(data2["trip_duration"]/60,0)
bins=[0,20,60,np.inf]
names=['0-20 Mins','21-60 Mins','>60 Mins']
data2["trip_bucket"]=pd.cut(data2["trip_duration"],bins,labels=names)
data2["hapm"] = data2["harsh_acc"]/data2["duration_mins"]
data2["hdpm"] = data2["harsh_dec"]/data2["duration_mins"]
data2["overspeed"] = data2["overspeed"]/data2["duration_mins"]
data2["harsh_events"] = data2["harsh_acc"] + data2["harsh_dec"]
data2["harsh_events_pm"] = data2["hapm"] + data2["hdpm"]
data3 = data2[["harsh_events", "trip_distance", "overspeed", "trip_duration"]]
data4 = data3.dropna(inplace=False)
X = data4.iloc[:, [0, 2]].values
data2 = df_updated.groupby(["trip_id", "Driver Type"]).agg(trip_duration=('Time (s)',np.max),
                                                           trip_distance=('distance_km',np.max),
                                                           max_speed=('Speed_kph',np.max), 
                                                           overspeed = ("overspeed", np.sum),
                                                           harsh_acc = ("harsh_acc", np.sum),
                                                           harsh_dec = ("harsh_dec", np.sum))
data2["duration_mins"] = round(data2["trip_duration"]/60,0)
bins=[0,20,60,np.inf]
names=['0-20 Mins','21-60 Mins','>60 Mins']
data2["trip_bucket"]=pd.cut(data2["trip_duration"],bins,labels=names)
data2["hapm"] = data2["harsh_acc"]/data2["duration_mins"]
data2["hdpm"] = data2["harsh_dec"]/data2["duration_mins"]
data2["overspeed%"] = data2["overspeed"]/data2["duration_mins"]
data2["harsh_events"] = data2["harsh_acc"] + data2["harsh_dec"]
data2["harsh_events_pm"] = data2["hapm"] + data2["hdpm"]
data2["harsh_events%"] = data2["harsh_events_pm"]/data2["duration_mins"]
data3 = data2[["harsh_events%", "trip_distance", "overspeed%", "trip_duration"]]
data4 = data3.dropna(inplace=False)
X = data4.iloc[:, [0, 3]].values
data3.head(2)






import scipy.cluster.hierarchy as sch
dendrogram = sch.dendrogram(sch.linkage(X, method = 'ward'))
plt.title('Dendrogram')
plt.xlabel('Harsh Events')
plt.ylabel('Euclidean distances')
plt.show()
from sklearn.cluster import AgglomerativeClustering
hc = AgglomerativeClustering(n_clusters = 3, metric = 'euclidean', linkage = 'ward')
y_hc = hc.fit_predict(X)
plt.scatter(X[y_hc == 0, 0], X[y_hc == 0, 1], s = 100, c = 'red', label = 'Aggressive')
plt.scatter(X[y_hc == 1, 0], X[y_hc == 1, 1], s = 100, c = 'blue', label = 'Calm')
plt.scatter(X[y_hc == 2, 0], X[y_hc == 2, 1], s = 100, c = 'green', label = 'Rational')
plt.title('Trip clusters')
plt.xlabel('Harsh Events%')
plt.ylabel('trip_duration (s)')
plt.legend()
plt.show()
import plotly.express as px
fig = px.scatter_3d(data4, x='harsh_events', y='trip_distance', z='overspeed',
                   color='trip_duration')
fig.show()
ext_data = pd.read_csv("trajectories_to_publish.csv")
ext_data2 = ext_data.head(200000)
ext_data.tail()
%%time
ext_data.describe()
extract_gdf = gpd.GeoDataFrame(ext_data2, geometry=gpd.points_from_xy(ext_data2.longitude, ext_data2.latitude), crs="EPSG:4326")
extract_gdf.crs
ext_data3 = extract_gdf.within(shapefile_dc)
extract_gdf = extract_gdf.to_crs("EPSG:4326")
ext_join_left_df = extract_gdf.sjoin_nearest(shapefile_tn_gdf, how="left", distance_col="Distance")
distance_col="Distances"
ext_join_left_df.head()
print("Minimum distance to shapefile record: ", ext_join_left_df["Distance"].min())
ext_data2.shape
df_updated.head(2)
df_updated.columns




data2 = df_updated.groupby(["trip_id", "Driver Type",
                           "dow", "hour", "surface", "highway"]).agg(trip_duration=('Time (s)',np.max),
                                                           trip_distance=('distance_km',np.max),
                                                           max_speed=('Speed_kph',np.max), 
                                                           overspeed = ("overspeed", np.sum),
                                                           harsh_acc = ("harsh_acc", np.sum),
                                                           harsh_dec = ("harsh_dec", np.sum))
data2["duration_mins"] = round(data2["trip_duration"]/60,0)
bins=[0,20,60,np.inf]
names=['0-20 Mins','21-60 Mins','>60 Mins']
data2["trip_bucket"]=pd.cut(data2["trip_duration"],bins,labels=names)
data2["hapm"] = data2["harsh_acc"]/data2["duration_mins"]
data2["hdpm"] = data2["harsh_dec"]/data2["duration_mins"]
data2["overspeed%"] = data2["overspeed"]/data2["duration_mins"]
data2["harsh_events"] = data2["harsh_acc"] + data2["harsh_dec"]
data2["harsh_events_pm"] = data2["hapm"] + data2["hdpm"]
data2["harsh_events%"] = data2["harsh_events_pm"]/data2["duration_mins"]

data3 = data2[["harsh_events%", "trip_distance", "overspeed%", "trip_duration"]]
data4 = data3.dropna(inplace=False)
X = data4.iloc[:, [0, 3]].values
data2 = df_updated.groupby(["trip_id", "Driver Type", "dow"]).agg(trip_duration=('Time (s)',np.max),
                                                           trip_distance=('distance_km',np.max),
                                                           max_speed=('Speed_kph',np.max), 
                                                                  avg_speed=('Speed_kph',np.mean), 
                                                           overspeed = ("overspeed", np.sum),
                                                           harsh_acc = ("harsh_acc", np.sum),
                                                           harsh_dec = ("harsh_dec", np.sum))
data2["duration_mins"] = round(data2["trip_duration"]/60,0)
bins=[0,20,60,np.inf]
names=['0-20 Mins','21-60 Mins','>60 Mins']
data2["trip_bucket"]=pd.cut(data2["trip_duration"],bins,labels=names)
data2["hapm"] = data2["harsh_acc"]/data2["duration_mins"]
data2["hdpm"] = data2["harsh_dec"]/data2["duration_mins"]
data2["overspeed%"] = data2["overspeed"]/data2["duration_mins"]
data2["harsh_events"] = data2["harsh_acc"] + data2["harsh_dec"]
data2["harsh_events_pm"] = data2["hapm"] + data2["hdpm"]
data2["harsh_events%"] = data2["harsh_events_pm"]/data2["duration_mins"]
data3 = data2.reset_index(inplace=False)
data3.head(2)





tbl = pd.pivot_table(data2,
    values="overspeed%",
    index="Driver Type",
    columns="dow",
    aggfunc= 'mean',
    fill_value='',
    margins = False,
    dropna = True,
    margins_name = 'All',
    observed= False,
    sort= True,
)
column_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
table = tbl.reindex(column_order, axis=1)
table
tbl = pd.pivot_table(data2,
    values="avg_speed",
    index="Driver Type",
    columns="dow",
    aggfunc= 'median',
    fill_value='',
    margins = False,
    dropna = True,
    margins_name = 'All',
    observed= False,
    sort= True,
)
column_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
table = tbl.reindex(column_order, axis=1)
table
data2 = df_updated.groupby(["trip_id", "Driver Type", "highway"]).agg(trip_duration=('Time (s)',np.max),
                                                           trip_distance=('distance_km',np.max),
                                                           max_speed=('Speed_kph',np.max), 
                                                            avg_speed=('Speed_kph',np.mean), 
                                                           overspeed = ("overspeed", np.sum),
                                                           harsh_acc = ("harsh_acc", np.sum),
                                                           harsh_dec = ("harsh_dec", np.sum))
data2["duration_mins"] = round(data2["trip_duration"]/60,0)
bins=[0,20,60,np.inf]
names=['0-20 Mins','21-60 Mins','>60 Mins']






data2["trip_bucket"]=pd.cut(data2["trip_duration"],bins,labels=names)
data2["hapm"] = data2["harsh_acc"]/data2["duration_mins"]
data2["hdpm"] = data2["harsh_dec"]/data2["duration_mins"]
data2["overspeed%"] = data2["overspeed"]/data2["duration_mins"]
data2["harsh_events"] = data2["harsh_acc"] + data2["harsh_dec"]
data2["harsh_events_pm"] = data2["hapm"] + data2["hdpm"]
data2["harsh_events%"] = data2["harsh_events_pm"]/data2["duration_mins"]
data3 = data2.reset_index(inplace=False)
pd.pivot_table(data2,
    values="avg_speed",
    index="highway",
    columns="Driver Type",
    aggfunc= 'median',
    fill_value='',
    sort = True)
pd.pivot_table(data2,
    values="overspeed%",
    index="highway",
    columns="Driver Type",
    aggfunc= 'median',
    fill_value='',
    sort = True)
data2 = df_updated.groupby(["trip_id", "Driver Type", "hour"]).agg(trip_duration=('Time (s)',np.max),
                                                           trip_distance=('distance_km',np.max),
                                                           max_speed=('Speed_kph',np.max), 
                                                            avg_speed=('Speed_kph',np.mean),
                                                           overspeed = ("overspeed", np.sum),
                                                           harsh_acc = ("harsh_acc", np.sum),
                                                           harsh_dec = ("harsh_dec", np.sum))
data2["duration_mins"] = round(data2["trip_duration"]/60,0)
bins=[0,20,60,np.inf]
names=['0-20 Mins','21-60 Mins','>60 Mins']
data2["trip_bucket"]=pd.cut(data2["trip_duration"],bins,labels=names)
data2["hapm"] = data2["harsh_acc"]/data2["duration_mins"]
data2["hdpm"] = data2["harsh_dec"]/data2["duration_mins"]
data2["overspeed%"] = data2["overspeed"]/data2["duration_mins"]
data2["harsh_events"] = data2["harsh_acc"] + data2["harsh_dec"]
data2["harsh_events_pm"] = data2["hapm"] + data2["hdpm"]
data2["harsh_events%"] = data2["harsh_events_pm"]/data2["duration_mins"]







data3 = data2.reset_index(inplace=False)
data3.head(2)
pd.pivot_table(data2,
    values="avg_speed",
    index="hour",
    columns="Driver Type",
    aggfunc= 'median',
    fill_value='')
pd.pivot_table(data2,
    values="overspeed%",
    index="hour",
    columns="Driver Type",
    aggfunc= 'median',
    fill_value='')
data2 = df_updated.groupby(["trip_id", "Driver Type", "surface"]).agg(trip_duration=('Time (s)',np.max),
                                                           trip_distance=('distance_km',np.max),
                                                           max_speed=('Speed_kph',np.max), 
                                                           avg_speed=('Speed_kph',np.mean), 
                                                           overspeed = ("overspeed", np.sum),
                                                           harsh_acc = ("harsh_acc", np.sum),
                                                           harsh_dec = ("harsh_dec", np.sum))
data2["duration_mins"] = round(data2["trip_duration"]/60,0)
bins=[0,20,60,np.inf]
names=['0-20 Mins','21-60 Mins','>60 Mins']
data2["trip_bucket"]=pd.cut(data2["trip_duration"],bins,labels=names)
data2["hapm"] = data2["harsh_acc"]/data2["duration_mins"]
data2["hdpm"] = data2["harsh_dec"]/data2["duration_mins"]
data2["overspeed%"] = data2["overspeed"]/data2["duration_mins"]
data2["harsh_events"] = data2["harsh_acc"] + data2["harsh_dec"]
data2["harsh_events_pm"] = data2["hapm"] + data2["hdpm"]
data2["harsh_events%"] = data2["harsh_events_pm"]/data2["duration_mins"]
data3 = data2.reset_index(inplace=False)
data3.head(2)







pd.pivot_table(data2,
    values="harsh_events%",
    index="surface",
    columns="Driver Type",
    aggfunc= 'median',
    fill_value='')
pd.pivot_table(data2,
    values="avg_speed",
    index="surface",
    columns="Driver Type",
    aggfunc= 'median',
    fill_value='')
data3.to_csv("Scoring_data.csv", index= False)